In [1]:
from optclaw.config import get_app_config
from optclaw.models import create_chat_model
from langchain.agents import create_agent

In [ ]:
c = get_app_config()

TitleConfig(enabled=True, max_words=6, max_chars=60, model_name=None, prompt_template='Generate a concise title (max {max_words} words) for this conversation.\nUser: {user_msg}\nAssistant: {assistant_msg}\n\nReturn ONLY the title, no quotes, no explanation.')

In [24]:
model = create_chat_model(thinking_enabled=True)

In [26]:
# 创建Agent
agent = create_agent(
    model=model,
    system_prompt="你好"
)

# 正确调用
response = agent.invoke({
    "messages": [{"role": "user", "content": "今天天气怎么样？"}]
})

print(response)

{'messages': [HumanMessage(content='今天天气怎么样？', additional_kwargs={}, response_metadata={}, id='33e2ea99-830b-4e87-925b-0aed2466cd81'), AIMessage(content='我没办法实时获取当地的天气信息哦~ 你可以通过手机自带的天气APP、微信或支付宝的天气小程序，或者搜索引擎搜索“XX地区今天天气”，就能查到精准的天气情况啦！', additional_kwargs={'refusal': None, 'reasoning_content': '用户问今天天气怎么样，我需要先说明无法实时获取天气信息，然后告诉用户可以通过哪些方式查询，比如天气APP、搜索引擎或者天气预报网站，这样用户就能自己查到准确的天气了。首先要礼貌回应，然后给出实用的建议，确保用户清楚怎么操作。'}, response_metadata={'token_usage': {'completion_tokens': 107, 'prompt_tokens': 42, 'total_tokens': 149, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 62, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'deepseek', 'model_name': 'doubao-seed-1-8-251228', 'system_fingerprint': None, 'id': '021777696999881c43b249df4b5023f4c24a0f20d5a332117038f', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019de6ff-892b-7fd1-b7f6-

In [ ]:
from optclaw.agents.factory import create_optclaw_agent

In [ ]:
from optclaw.agents.middlewares.thread_data_middleware import ThreadDataMiddleware
from optclaw.agents.middlewares.uploads_middleware import UploadsMiddleware
from optclaw.sandbox.middleware import SandboxMiddleware
# from optclaw.sandbox import get_sandbox_provider

In [63]:
tt = SandboxMiddleware()

In [13]:
import threading
import time

# 创建事件
event = threading.Event()

def worker():
    print("工作线程：等待主线程信号...")
    event.wait()  # 阻塞，等待 set()
    print("工作线程：收到信号！开始执行任务")

# 启动工作线程
t = threading.Thread(target=worker)
t.start()

# time.sleep(2)  # 模拟主线程做其他事

工作线程：等待主线程信号...


In [12]:
print("\n主线程：发送开始信号")
event.set()  # 触发事件，唤醒等待的线程

# t.join()


主线程：发送开始信号


In [14]:
import threading
import time

event = threading.Event()

def work_once(idx):
    print(f"第{idx}轮：开始加载...")
    time.sleep(2)
    print(f"第{idx}轮：加载完成，set")
    event.set()

if __name__ == "__main__":
    # 第一轮
    event.clear()
    t1 = threading.Thread(target=work_once, args=(1,))
    t1.start()
    event.wait()
    print("第一轮结束\n")

    # 第二轮必须再 clear！
    event.clear()
    t2 = threading.Thread(target=work_once, args=(2,))
    t2.start()
    event.wait()
    print("第二轮结束")

第1轮：开始加载...
第1轮：加载完成，set
第一轮结束

第2轮：开始加载...
第2轮：加载完成，set
第二轮结束


In [7]:
import concurrent.futures
import atexit
import time

# 1. 创建线程池（你原来的代码）
_SYNC_MEMORY_UPDATER_EXECUTOR = concurrent.futures.ThreadPoolExecutor(
    max_workers=4,
    thread_name_prefix="memory-updater-sync",
)

# 2. 程序退出时自动关闭线程池
atexit.register(lambda: _SYNC_MEMORY_UPDATER_EXECUTOR.shutdown(wait=False))

# ------------------------------------------------
# 下面是你自己的业务函数，随便定义
def update_memory(data):
    """模拟后台更新内存/缓存/数据的任务"""
    print(f"后台开始处理：{data}")
    time.sleep(2)  # 模拟耗时操作
    print(f"后台处理完成：{data}")

# ------------------------------------------------
# 使用方法 1：最常用 —— 提交任务，后台自动跑
future=_SYNC_MEMORY_UPDATER_EXECUTOR.submit(update_memory, "用户1数据")
future.result()
future=_SYNC_MEMORY_UPDATER_EXECUTOR.submit(update_memory, "用户2数据")
future.result()
future=_SYNC_MEMORY_UPDATER_EXECUTOR.submit(update_memory, "用户3数据")
future.result()
future=_SYNC_MEMORY_UPDATER_EXECUTOR.submit(update_memory, "用户4数据")
future.result()
future=_SYNC_MEMORY_UPDATER_EXECUTOR.submit(update_memory, "用户5数据")
future.result()

# print("主程序继续运行，不会被阻塞！")

后台开始处理：用户1数据
后台处理完成：用户1数据
后台开始处理：用户2数据
后台处理完成：用户2数据
后台开始处理：用户3数据
后台处理完成：用户3数据
后台开始处理：用户4数据
后台处理完成：用户4数据
后台开始处理：用户5数据
后台处理完成：用户5数据


In [4]:
from dataclasses import dataclass, field


class Message:
    ai_messages: list = []  # ❌ 危险！所有实例共用同一个列表！

In [ ]:
msg1 = Message()
msg2 = Message()

msg1.ai_messages.append("hello")

print(msg1.ai_messages)  # ['hello']
print(msg2.ai_messages)  # ['hello']  ❌ 居然也变了！

AttributeError: 'Message' object has no attribute 'copy'